In [31]:
from db_connector import list_all_tables, load_table_to_df
import pandas as pd
import numpy as np

# DB 내의 모든 테이블 로드
all_tables = list_all_tables()
all_dfs = {}
for table in all_tables:
    try:
        all_dfs[table] = load_table_to_df(table)
    except Exception as e:
        pass
print("모든 데이터프레임 로드 완료!")

모든 데이터프레임 로드 완료!


In [41]:
# 1) 식단 조건 생성 함수
def get_meal_conditions(run_id, all_dfs):
    # all_dfs에서 recommendation_runs 데이터프레임을 가져옵니다.
    df = all_dfs.get('recommendation_runs')
    
    if df is None:
        print("Error: 'recommendation_runs' 데이터프레임을 찾을 수 없습니다.")
        return None, None, None
    
    # run_id가 일치하는 행을 찾습니다. (문자열/숫자 호환성을 위해 int 변환)
    try:
        run_id_int = int(run_id)
        run_data = df[df['run_id'] == run_id_int]
    except (ValueError, TypeError):
        run_data = df[df['run_id'] == run_id]
    
    if run_data.empty:
        print(f"Error: run_id {run_id}에 해당하는 데이터를 찾을 수 없습니다.")
        return None, None, None
    
    # 첫 번째 일치하는 행의 데이터를 추출합니다.
    row = run_data.iloc[0]
    
    user_id = row['user_id']
    target_meal_calories_kcal = row['target_meal_calories_kcal']
    target_meal_budget_krw = row['target_meal_budget_krw']
    
    return user_id, target_meal_calories_kcal, target_meal_budget_krw

In [42]:
get_meal_conditions(50005, all_dfs)

(np.int64(1), np.float64(1.0), np.int64(0))

In [43]:
# 2) 조건에 맞는 식단 생성 함수 (MILP)
import pulp
import uuid
import pandas as pd
import datetime

def generate_meal_candidates(user_id, target_meal_calories_kcal, target_meal_budget_krw, all_dfs):
    #ALL_DFS로부터 데이터 파싱
    foods_df = all_dfs['foods']
    user_prof_df = all_dfs['user_profiles']
    user_profile = user_prof_df[user_prof_df['user_id'] == user_id].iloc[0]
    goal_type = user_profile['goal_type']
    user_al = all_dfs.get('user_allergens', pd.DataFrame())
    food_al = all_dfs.get('food_allergens', pd.DataFrame())
    forbidden_foods = []
    # 알레르기 가진 식품 배제
    if not user_al.empty and not food_al.empty:
        my_allergens = user_al[user_al['user_id'] == user_id]['allergen_id'].tolist()
        forbidden_foods = food_al[food_al['allergen_id'].isin(my_allergens)]['food_id'].tolist()
    # GOAL_TYPE에 따른 한끼 요구 칼로리와 탄단지 비율
    meal_cal = target_meal_calories_kcal / 3
    if goal_type == 'bulk':
        min_cal, max_cal = meal_cal + 300, meal_cal + 500
        ratios = {'carbs': 0.5, 'protein': 0.3, 'fat': 0.2}
    elif goal_type == 'diet':
        min_cal, max_cal = meal_cal - 500, meal_cal - 300
        ratios = {'carbs': 0.4, 'protein': 0.4, 'fat': 0.2}
    else:
        min_cal, max_cal = meal_cal * 0.9, meal_cal * 1.1
        ratios = {'carbs': 0.5, 'protein': 0.2, 'fat': 0.3}
    #예산 보다 비싼 식품 제외
    prob = pulp.LpProblem("Meal_Gen", pulp.LpMinimize)
    food_items = [f for f in foods_df.to_dict('records') 
                  if f['price_krw'] <= target_meal_budget_krw and f['food_id'] not in forbidden_foods]
    #제약조건
    food_vars = pulp.LpVariable.dicts("food", [f['food_id'] for f in food_items], 0, 1, pulp.LpBinary)
    prob += pulp.lpSum([f['price_krw'] * food_vars[f['food_id']] for f in food_items])
    prob += pulp.lpSum([food_vars[f['food_id']] for f in food_items]) >= 1
    prob += pulp.lpSum([food_vars[f['food_id']] for f in food_items]) <= 3
    prob += pulp.lpSum([f['price_krw'] * food_vars[f['food_id']] for f in food_items]) <= target_meal_budget_krw
    prob += pulp.lpSum([f['calories_kcal'] * food_vars[f['food_id']] for f in food_items]) >= min_cal
    prob += pulp.lpSum([f['calories_kcal'] * food_vars[f['food_id']] for f in food_items]) <= max_cal
    
    for macro, ratio in ratios.items():
        target_g = (meal_cal * ratio) / (9 if macro == 'fat' else 4)
        prob += pulp.lpSum([f[macro+'_g'] * food_vars[f['food_id']] for f in food_items]) >= target_g * 0.7
        prob += pulp.lpSum([f[macro+'_g'] * food_vars[f['food_id']] for f in food_items]) <= target_g * 1.3
        
    if 'meal_candidates' in all_dfs and not all_dfs['meal_candidates'].empty:
        next_cand_id = int(all_dfs['meal_candidates']['candidate_id'].max() + 1)
    else:
        next_cand_id = 0
        
    if 'meal_candidate_items' in all_dfs and not all_dfs['meal_candidate_items'].empty:
        next_item_id = int(all_dfs['meal_candidate_items']['meal_candidate_item_id'].max() + 1)
    else:
        next_item_id = 0

    meal_candidates = []
    meal_candidate_items = []
    now_str = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    for _ in range(10):
        prob.solve(pulp.PULP_CBC_CMD(msg=0))
        if pulp.LpStatus[prob.status] == 'Optimal':
            selected_foods = [f for f in food_items if food_vars[f['food_id']].varValue == 1]
            cand_id = next_cand_id
            next_cand_id += 1
            
            t_price = sum(f['price_krw'] for f in selected_foods)
            t_cal = sum(f['calories_kcal'] for f in selected_foods)
            t_protein = sum(f['protein_g'] for f in selected_foods)
            t_fat = sum(f['fat_g'] for f in selected_foods)
            t_carbs = sum(f['carbs_g'] for f in selected_foods)
            
            food_ids_sorted = sorted([str(f['food_id']) for f in selected_foods])
            food_names_sorted = all_dfs['foods'].set_index('food_id')['food_name'].to_dict()
            
            meal_candidates.append({
                'candidate_id': cand_id, 'candidate_name': ", ".join(food_names_sorted[int(food_id)] for food_id in food_ids_sorted),
                'candidate_fingerprint': None, 'fingerprint_version': None,
                'meal_type': None, 'meal_channel': None,
                'total_price_krw': t_price, 'total_calories_kcal': t_cal,
                'total_protein_g': t_protein, 'total_fat_g': t_fat, 'total_carbs_g': t_carbs,
                'generation_source': None, 'is_active': None,
                'created_at': now_str, 'updated_at': now_str
            })
            
            for idx, f in enumerate(selected_foods, 1):
                meal_candidate_items.append({
                    'meal_candidate_item_id': next_item_id, 'candidate_id': cand_id,
                    'food_id': f['food_id'], 'quantity_g': None,
                    'quantity_label': None, 'quantity_bucket': None,
                    'item_order': None, 'item_price_krw': f['price_krw'],
                    'item_calories_kcal': f['calories_kcal'], 'item_protein_g': f['protein_g'],
                    'item_fat_g': f['fat_g'], 'item_carbs_g': f['carbs_g']
                })
                next_item_id += 1
            prob += pulp.lpSum([food_vars[f['food_id']] for f in selected_foods]) <= len(selected_foods) - 1
        else:
            break
    return meal_candidates, meal_candidate_items

In [44]:
meal_candidates, meal_candidate_items = generate_meal_candidates(
    user_id=1,
    target_meal_calories_kcal=2174,
    target_meal_budget_krw=13000,
    all_dfs=all_dfs
)

df1 = pd.DataFrame(meal_candidates)
df2 = pd.DataFrame(meal_candidate_items)

In [45]:
df1.head()

,candidate_id,candidate_name,candidate_fingerprint,fingerprint_version,meal_type,meal_channel,total_price_krw,total_calories_kcal,total_protein_g,total_fat_g,total_carbs_g,generation_source,is_active,created_at,updated_at
0,50072,"촉촉한 구운란, 도시락",None,None,None,None,1260,670.10,28.00,23.40,82.00,None,None,2026-06-05 18:20:03,2026-06-05 18:20:03
1,50073,"식물성 대체계란 드라이 믹스, 도시락",None,None,None,None,1260,670.20,28.00,18.90,100.00,None,None,2026-06-05 18:20:03,2026-06-05 18:20:03
2,50074,"프로틴우노바 코코넛 밀크, 도시락",None,None,None,None,1320,673.95,29.00,22.80,91.00,None,None,2026-06-05 18:20:03,2026-06-05 18:20:03
3,50075,"뷰티니 프로틴 밀크초코볼, 도시락",None,None,None,None,1320,720.90,30.00,27.00,86.00,None,None,2026-06-05 18:20:03,2026-06-05 18:20:03
4,50076,"피킷 프로틴쉐이크 크런치초코밀크, 도시락",None,None,None,None,1320,728.00,39.23,20.54,90.56,None,None,2026-06-05 18:20:03,2026-06-05 18:20:03


In [46]:
df2.head()

,meal_candidate_item_id,candidate_id,food_id,quantity_g,quantity_label,quantity_bucket,item_order,item_price_krw,item_calories_kcal,item_protein_g,item_fat_g,item_carbs_g
0,50120,50072,3517,None,None,None,None,620,570.0,18.0,18.0,78.0
1,50121,50072,15215,None,None,None,None,640,100.1,10.0,5.4,4.0
2,50122,50073,3517,None,None,None,None,620,570.0,18.0,18.0,78.0
3,50123,50073,15365,None,None,None,None,640,100.2,10.0,0.9,22.0
4,50124,50074,3517,None,None,None,None,620,570.0,18.0,18.0,78.0


In [13]:
all_dfs['meal_candidates'] = pd.concat(
    [all_dfs['meal_candidates'], df1], 
    axis=0,        # 0: 행 방향(아래로) 결합, 1: 열 방향(옆으로) 결합
    ignore_index=True # 인덱스를 0부터 새로 재정렬 (추천)
)
all_dfs['meal_candidate_items'] = pd.concat(
    [all_dfs['meal_candidate_items'], df2], 
    axis=0,        # 0: 행 방향(아래로) 결합, 1: 열 방향(옆으로) 결합
    ignore_index=True # 인덱스를 0부터 새로 재정렬 (추천)
)

In [14]:
all_dfs['meal_candidates']

,candidate_id,candidate_name,candidate_fingerprint,fingerprint_version,meal_type,meal_channel,total_price_krw,total_calories_kcal,total_protein_g,total_fat_g,total_carbs_g,generation_source,is_active,created_at,updated_at
0,1,리셋프로틴쉐이크 군고구마,single-food:dinner:1066,v1,dinner,home_meal,4950,1900.0,250.00,29.00,170.00,seed,True,2026-06-03 04:16:30.897231,2026-06-03 04:16:30.897231
1,2,큐브닭가슴살S,single-food:dinner:30,v1,dinner,home_meal,8500,1150.0,250.00,10.00,20.00,seed,True,2026-06-03 04:16:30.936090,2026-06-03 04:16:30.936090
2,3,슈레드 스팀닭가슴살,single-food:dinner:32,v1,dinner,home_meal,8500,1160.0,250.00,13.00,7.00,seed,True,2026-06-03 04:16:30.976474,2026-06-03 04:16:30.976474
3,4,닭가슴살 채,single-food:dinner:34,v1,dinner,home_meal,8500,1200.0,250.00,14.00,12.00,seed,True,2026-06-03 04:16:31.020665,2026-06-03 04:16:31.020665
4,5,닭가슴살 채(냉동),single-food:dinner:33,v1,dinner,home_meal,8500,1200.0,250.00,14.00,12.00,seed,True,2026-06-03 04:16:31.051578,2026-06-03 04:16:31.051578
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
725,726,"베어벨스(BAREBELLS) 프로틴 밀크쉐이크 바닐라향, 도시락",None,None,None,None,1320,761.0,42.00,22.60,91.00,None,None,2026-06-04 23:25:10,2026-06-04 23:25:10
726,727,"베어벨스(BAREBELLS) 프로틴 밀크쉐이크 딸기향, 도시락",None,None,None,None,1320,761.0,42.00,22.60,91.00,None,None,2026-06-04 23:25:10,2026-06-04 23:25:10
727,728,"작심랩 단백질쉐이크 딸기우유맛, 뽀로로친구들 퍼즐비스킷 계란맛",None,None,None,None,1340,738.1,46.45,21.39,90.53,None,None,2026-06-04 23:25:10,2026-06-04 23:25:10
728,729,"초코밀크프로틴볼, 아이밀냠냠계란볼",None,None,None,None,1340,723.0,31.20,29.00,84.00,None,None,2026-06-04 23:25:10,2026-06-04 23:25:10


In [ ]:
# 파이프라인 메인 함수
import datetime

def process_recommendation_pipeline(run_id, all_dfs):
    user_id, target_cals, budget = get_meal_conditions(run_id, all_dfs)
    if user_id is None: return None
    
    # 1. 식단 생성
    meal_cands, meal_items = generate_meal_candidates(user_id, target_cals, budget, all_dfs)
        
    # 2. 스코어링 (LightFM에 meal_candidate_items 전달)
    lightfm_scores = calculate_lightfm_scores(user_id, meal_cands, meal_items, all_dfs)
    xgboost_probs = calculate_xgboost_probabilities(user_id, meal_cands, all_dfs)
    mmr_pens = calculate_mmr_penalties(user_id, meal_cands, meal_items, all_dfs)
    
    recommendation_candidates_data = []
    now_str = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    for i, cand in enumerate(meal_cands):
        cid = cand['candidate_id']
        lfm_score = lightfm_scores.get(cid, 0.0)
        xgb_prob = xgboost_probs.get(cid, 0.0)
        mmr_p, repeat_p = mmr_pens.get(cid, (0.0, 0.0))
        final_score = (xgb_prob * 0.5) + (lfm_score * 0.3) - mmr_p - repeat_p
        
        recommendation_candidates_data.append({
            'recommendation_candidate_id': None, 'run_id': None, 'candidate_id': cid,
            'milp_feasible': True, 'milp_rank': i + 1, 'rule_score': 0.0,
            'lightfm_score': lfm_score, 'xgboost_probability': xgb_prob,
            'mmr_penalty': mmr_p, 'repeat_food_penalty': repeat_p,
            'repeat_combo_penalty': 0.0, 'final_score': final_score, 'final_rank': None,
            'feature_snapshot': None, 'score_breakdown': None, 'was_selected': False,
            'selected_at': None, 'created_at': now_str
        })
    
    recommendation_candidates_data.sort(key=lambda x: x['final_score'], reverse=True)
    for rank, item in enumerate(recommendation_candidates_data, 1):
        item['final_rank'] = rank
        
    return {
        'new_meal_candidates': meal_cands,
        'new_meal_candidate_items': meal_items,
        'recommendation_candidates_data': recommendation_candidates_data
    }